In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class ConvBlock(nn.Module):
  super().__init__()
  def __init__(self, in_channels, out_channels, stride=1):
    self.is_last = False

    # first 3x3 convolution (may downsample if stride > 1)
    self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False) 
    self.bn1 = nn.BatchNorm2d(out_channels)
    self.relu = nn.ReLU(inplace=True)
    self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False) # stride = 1
    self.bn2 = nn.BatchNorm2d(out_channels)

    # identity shortcut by default
    self.shortcut = nn.Sequential() 

    # projection shortcut to match feature-map size or channel count
    if stride != 1 or in_channels != out_channels: # if (h, w) changed or channels dims changed
      self.shortcut = nn.Sequential(
          nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
          nn.BatchNorm2d(out_channels)
      )

    def forward(self, x):
      out = self.conv1(x)
      out = self.bn1(out)
      out = self.relu(out)
      out = self.conv2(out)
      out = self.bn2(out)
      # residual connection
      out += self.shortcut(x)
      out = self.relu(out)
      return out